# M4 L4: ANN and Vector Database Internals

**Build Flat / IVF / PQ / HNSW indexes from scratch.  Benchmark them.**

Duration: 90 min.  No API keys.

## Plain English (the 12-year-old version)

**ANN** stands for "Approximate Nearest Neighbors" — a fancy way of saying "really fast search that's almost exactly right."

Imagine you have 1 million cards, each with a different drawing. Someone shows you a new drawing and asks "which of your million cards looks most like this?" The slow way: look at every card. The fast way: use clues to skip most of them.

There are four big tricks (families): **flat** (look at all of them — slow but exact), **IVF** (sort the cards into buckets; only check the buckets that look promising), **PQ** (replace each card with a tiny code; compare codes), and **HNSW** (build a web of "similar" connections; walk the web). Each trick makes a different deal: faster but less exact, or uses less memory but less exact.

## The 5 things to know

1. **Flat is the ground truth** — every ANN is measured against it.  If flat doesn't return 100%, your distance function is wrong.
2. **HNSW is the production default** — best recall-to-speed ratio; M and efSearch are the two knobs.
3. **IVF trades recall for speed** — n_probe is the knob; sweep it.
4. **PQ trades recall for memory** — 8-32× compression at 5-15pp recall hit.
5. **Always validate against flat on a golden set** — never ship an ANN you haven't benchmarked.

In [1]:
# === Standard setup ===
import sys, importlib.util, time
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm04-ann' / 'L4-ann-internals.py'
_spec = importlib.util.spec_from_file_location('L4', _p)
_L4 = importlib.util.module_from_spec(_spec)
sys.modules['L4'] = _L4
_spec.loader.exec_module(_L4)

print(f'Setup OK. Lesson module: {_L4.LESSON_NUMBER}')

Setup OK. Lesson module: M4.L4


In [2]:
# Run the 10 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L4, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

Running 25 tests...

  PASS  test_ann_math_invariants
  PASS  test_flat_recall_is_perfect
  PASS  test_flat_search_on_known_neighbor
  PASS  test_flat_search_top_k_sorted
  PASS  test_hnsw_graph_is_symmetric
  PASS  test_hnsw_recall_above_random
  PASS  test_hnsw_returns_high_scoring_neighbors
  PASS  test_hnsw_returns_k_results
  PASS  test_index_comparison_table_constructed
  PASS  test_ivf_postings_partition_indices
  PASS  test_ivf_returns_k_results
  PASS  test_ivf_search_returns_real_neighbors
  PASS  test_make_corpus_dim_matches_request
  PASS  test_make_corpus_is_deterministic_per_seed
  PASS  test_make_corpus_normalized
  PASS  test_make_queries_come_from_corpus
  PASS  test_make_queries_count
  PASS  test_pq_codes_in_valid_range
  PASS  test_pq_compresses_vector
  PASS  test_pq_returns_k_results
  PASS  test_recall_at_k_full_overlap
  PASS  test_recall_at_k_handles_short_retrieved
  PASS  test_recall_at_k_partial
  PASS  test_recall_at_k_perfect
  PASS  test_recall_at_k_zero_

In [3]:
# === Section 1: build a synthetic corpus and queries ===
import numpy as np
vectors, ids = _L4.make_corpus(_L4.N_VECTORS, _L4.DIM, seed=_L4.SEED)
queries = _L4.make_queries(vectors, _L4.N_QUERIES, seed=_L4.SEED + 1)
print(f'Corpus: {vectors.shape[0]} vectors, dim={vectors.shape[1]}')
print(f'Queries: {queries.shape[0]} vectors (random subset of corpus)')
print(f'Norms: min={np.linalg.norm(vectors, axis=1).min():.3f}, max={np.linalg.norm(vectors, axis=1).max():.3f} (should be 1.0)')

Corpus: 2000 vectors, dim=64
Queries: 50 vectors (random subset of corpus)
Norms: min=1.000, max=1.000 (should be 1.0)


In [4]:
# === Section 2: flat (ground truth) ===
flat = _L4.FlatIndex(vectors=vectors, ids=ids)
t0 = time.perf_counter()
truth = [flat.search(q, k=_L4.TOP_K) for q in queries]
flat_ms = (time.perf_counter() - t0) / _L4.N_QUERIES * 1000
print(f'Flat avg latency: {flat_ms:.2f} ms/query')
print('Flat top-1 example:', truth[0][0], 'score=', truth[0][1])

Flat avg latency: 0.13 ms/query
Flat top-1 example: ('v00358', 0.9999999403953552) score= ('v01786', 0.8549505472183228)


In [5]:
# === Section 3: IVF-lite ===
ivf = _L4.build_ivf_lite(vectors, ids, n_lists=_L4.N_LISTS_IVF, seed=_L4.SEED)
t0 = time.perf_counter()
ivf_hits = [ivf.search(q, k=_L4.TOP_K) for q in queries]
ivf_ms = (time.perf_counter() - t0) / _L4.N_QUERIES * 1000
ivf_recall = float(np.mean([_L4.recall_at_k(h, t, _L4.TOP_K) for h, t in zip(ivf_hits, truth)]))
print(f'IVF avg latency:   {ivf_ms:.2f} ms/query')
print(f'IVF mean recall@10: {ivf_recall:.3f}')
print(f'IVF speedup vs flat: {flat_ms/ivf_ms:.1f}x')

IVF avg latency:   0.11 ms/query
IVF mean recall@10: 0.592
IVF speedup vs flat: 1.1x


In [6]:
# === Section 4: HNSW-lite ===
hnsw = _L4.HNSWLiteIndex(M=_L4.M_HNSW, ef_construction=_L4.EF_CONSTRUCTION,
                          ef_search=_L4.EF_SEARCH, vectors=vectors, ids=ids)
t0 = time.perf_counter()
for i in range(vectors.shape[0]):
    hnsw.add(i)
build_s = time.perf_counter() - t0
t0 = time.perf_counter()
hnsw_hits = [hnsw.search(q, k=_L4.TOP_K) for q in queries]
hnsw_ms = (time.perf_counter() - t0) / _L4.N_QUERIES * 1000
hnsw_recall = float(np.mean([_L4.recall_at_k(h, t, _L4.TOP_K) for h, t in zip(hnsw_hits, truth)]))
print(f'HNSW build time:   {build_s:.2f} s')
print(f'HNSW avg latency:  {hnsw_ms:.2f} ms/query')
print(f'HNSW mean recall@10: {hnsw_recall:.3f}')
print(f'HNSW graph degree for node 500: {len(hnsw.graph[500])}')

HNSW build time:   0.14 s
HNSW avg latency:  1.50 ms/query
HNSW mean recall@10: 0.940
HNSW graph degree for node 500: 10


In [7]:
# === Section 5: PQ-lite ===
pq = _L4.build_pq_lite(vectors, n_subq=8, k=16, seed=_L4.SEED)
t0 = time.perf_counter()
pq_hits = [pq.search(vectors, ids, q, k=_L4.TOP_K) for q in queries]
pq_ms = (time.perf_counter() - t0) / _L4.N_QUERIES * 1000
pq_recall = float(np.mean([_L4.recall_at_k(h, t, _L4.TOP_K) for h, t in zip(pq_hits, truth)]))
print(f'PQ avg latency:  {pq_ms:.2f} ms/query')
print(f'PQ mean recall@10: {pq_recall:.3f}')
compression = (_L4.DIM * 4) / (8 * 4)
print(f'PQ compression:  {compression:.1f}x  (raw 256B -> 32B)')

PQ avg latency:  0.26 ms/query
PQ mean recall@10: 0.216
PQ compression:  8.0x  (raw 256B -> 32B)


In [8]:
# === Section 6: index comparison table ===
print(f'{"index":<8} {"recall@10":>10} {"avg ms":>10} {"speedup":>10}')
print('-' * 40)
print(f'{"flat":<8} {1.0:>10.3f} {flat_ms:>9.2f} {"1.0x":>10}')
print(f'{"ivf":<8} {ivf_recall:>10.3f} {ivf_ms:>9.2f} {flat_ms/ivf_ms:>9.1f}x')
print(f'{"hnsw":<8} {hnsw_recall:>10.3f} {hnsw_ms:>9.2f} {flat_ms/hnsw_ms:>9.1f}x')
print(f'{"pq":<8} {pq_recall:>10.3f} {pq_ms:>9.2f} {flat_ms/max(pq_ms,1e-9):>9.1f}x')

index     recall@10     avg ms    speedup
----------------------------------------
flat          1.000      0.13       1.0x
ivf           0.592      0.11       1.1x
hnsw          0.940      1.50       0.1x
pq            0.216      0.26       0.5x


In [9]:
# === Section 7: knob sweep -- n_probe on IVF ===
print('Sweeping n_probe on IVF:\n')
print(f'{"n_probe":>8} {"recall@10":>12} {"avg ms":>10}')
for np_ in (1, 2, 4, 8, 16):
    ivf.n_probe = np_
    t0 = time.perf_counter()
    hits = [ivf.search(q, k=_L4.TOP_K) for q in queries]
    dt = (time.perf_counter() - t0) / _L4.N_QUERIES * 1000
    rec = float(np.mean([_L4.recall_at_k(h, t, _L4.TOP_K) for h, t in zip(hits, truth)]))
    print(f'{np_:>8} {rec:>12.3f} {dt:>9.2f}')

Sweeping n_probe on IVF:

 n_probe    recall@10     avg ms
       1        0.296      0.05
       2        0.442      0.07
       4        0.592      0.09
       8        0.830      0.15
      16        1.000      0.26


In [10]:
# === Section 8: knob sweep -- ef_search on HNSW ===
print('Sweeping ef_search on HNSW:\n')
print(f'{"ef_search":>10} {"recall@10":>12} {"avg ms":>10}')
for ef in (8, 16, 32, 64):
    hnsw.ef_search = ef
    t0 = time.perf_counter()
    hits = [hnsw.search(q, k=_L4.TOP_K) for q in queries]
    dt = (time.perf_counter() - t0) / _L4.N_QUERIES * 1000
    rec = float(np.mean([_L4.recall_at_k(h, t, _L4.TOP_K) for h, t in zip(hits, truth)]))
    print(f'{ef:>10} {rec:>12.3f} {dt:>9.2f}')

Sweeping ef_search on HNSW:

 ef_search    recall@10     avg ms
         8        0.752      0.45
        16        0.872      0.78
        32        0.940      1.30


        64        0.984      1.92


## Architecture (Mermaid)

```mermaid
flowchart TB
    A[N vectors, dim D] --> B{index family}
    B -->|flat| F[FlatIndex: matrix @ query]
    B -->|IVF| I1[k-means centroids]
    I1 --> I2[posting lists per centroid]
    I2 --> I3[at query: top n_probe centroids]
    I3 --> I4[brute force within probe set]
    B -->|HNSW| H1[for each node: M nearest by brute force]
    H1 --> H2[bidirectional edges]
    H2 --> H3[at query: greedy walk from entry]
    B -->|PQ| P1[split into n_subq sub-vectors]
    P1 --> P2[k-means per sub]
    P2 --> P3[codebook: n_subq * k centroids]
    P3 --> P4[codes: n_subq ints per vector]
    F --> E[sorted top-K]
    I4 --> E
    H3 --> E
    P4 --> E
```

**Read as:** Pick the family by the constraint. Flat for <100K. HNSW for 100K-10M with low latency. IVF-PQ for 10M+ with memory pressure.

## Reflect (10 min)

1. Why is brute force KNN the ground truth for ANN validation?
2. What's the difference between IVF, PQ, and HNSW? Which one would you pick for a 50M-vector index on a single machine with 32GB RAM?
3. The n_probe sweep shows recall grows with n_probe. What's the trade-off?
4. The ef_search sweep shows the same. Why does the latency grow linearly?
5. (Stretch) Implement an `HNSWLiteIndex.search_with_prefilter(candidate_ids, q, k)` that only visits nodes in the candidate set.

---

**Read next:**
- `lessons/m05-ingestion/L5-document-ingestion.ipynb` — idempotent ingestion, content-hash dedupe
- `lessons/m07-retrieval/L7-retrieval-ranking.ipynb` — BM25, dense, RRF, MMR
- `SYLLABUS.md` — the 18 modules and 21-day plan